In [1]:
### Imports and Dataset path

In [2]:
import pandas as pd
import glob, os

# Dataset folder 
DATA_DIR = r"C:\Users\ramku\OneDrive\Documents\Research Practicum\Execution\Reopened bugs dataset"

files = sorted(glob.glob(os.path.join(DATA_DIR, "joined_project_dataframe_*.csv")))
print("Number of project files found:", len(files))   # expected is 47

Number of project files found: 47


In [3]:
### To look into the structure (columns and Target)

In [4]:
sample = pd.read_csv(files[0], low_memory=False)
print("Shape:", sample.shape)
print("\nColumns:")
for c in sample.columns:
    print("  -", c)

# The target variable:
print("\nTarget 'Reopened' unique values:", sample['Reopened'].unique())

Shape: (2086, 24)

Columns:
  - snapshot_key
  - changelog_Time
  - changelog_Weekday
  - changelog_Month_day
  - changelog_Month
  - changelog_Day_of_year
  - updated_component
  - updated_priority
  - updated_description
  - Number_of_comments
  - accumulated_comment_text
  - changelog_Priority_changed
  - Time_days
  - changelog_Last_status
  - Number_of_files_changed
  - snapshot_reporter
  - snapshot_assignee
  - Reopened
  - updated_description_size
  - accumulated_comment_size
  - reporter_experience
  - assignee_experience
  - description_pre_processed_text
  - comment_pre_processed_text

Target 'Reopened' unique values: [0 1]


In [5]:
### To load all the 47 projects and to build a per project summary

In [6]:
rows = []
for f in files:
    proj = os.path.basename(f).replace("joined_project_dataframe_", "").replace(".csv", "")
    df = pd.read_csv(f, usecols=["Reopened"], low_memory=False)  # only load the label = fast
    reopened = int((df["Reopened"] == 1).sum())
    non_reopened = int((df["Reopened"] == 0).sum())
    rows.append({
        "project": proj,
        "total_bugs": len(df),
        "non_reopened": non_reopened,
        "reopened": reopened,
        "reopened_pct": round(100 * reopened / len(df), 2)
    })

summary = pd.DataFrame(rows).sort_values("reopened", ascending=False).reset_index(drop=True)
summary

,project,total_bugs,non_reopened,reopened,reopened_pct
0,AMBARI,16600,15765,835,5.03
1,SPARK,12638,11918,720,5.70
2,CASSANDRA,7885,7317,568,7.20
3,FLINK,6464,5958,506,7.83
4,OFBIZ,4083,3692,391,9.58
5,HADOOP,6617,6252,365,5.52
6,GEODE,3854,3497,357,9.26
7,HAWQ,756,401,355,46.96
8,MNG,3166,2814,352,11.12
9,THRIFT,2524,2177,347,13.75


In [7]:
### To verify the total against the paper

In [8]:
print("Projects      :", len(summary))
print("Total bugs    :", summary["total_bugs"].sum())
print("Non-reopened  :", summary["non_reopened"].sum())
print("Reopened      :", summary["reopened"].sum())
print("Reopened %    : {:.2f}%".format(
      100 * summary["reopened"].sum() / summary["total_bugs"].sum()))

print("\nTagra et al. reported: 178,636 non-reopened + 9,993 reopened")

Projects      : 47
Total bugs    : 168881
Non-reopened  : 159995
Reopened      : 8886
Reopened %    : 5.26%

Tagra et al. reported: 178,636 non-reopened + 9,993 reopened


In [9]:
### Data Scarcity Issue
# This directly shows the data-scarcity gap, meaning some projects have only a handful of reopened bugs, 
# which is exactly why per-project models fail and why this cross-project idea could help.

In [10]:
print("Smallest projects by reopened-bug count:")
print(summary.nsmallest(5, "reopened")[["project","total_bugs","reopened"]].to_string(index=False))

print("\nLargest projects by reopened-bug count:")
print(summary.nlargest(5, "reopened")[["project","total_bugs","reopened"]].to_string(index=False))

# projects below 10-reopened filter threshold
below10 = summary[summary["reopened"] < 10]
print("\nProjects with < 10 reopened bugs:", len(below10))
print(below10[["project","reopened"]].to_string(index=False))

Smallest projects by reopened-bug count:
project  total_bugs  reopened
  BAHIR          61         5
 CTAKES         162         8
    ORC         268        12
PARQUET         651        16
   OODT         371        17

Largest projects by reopened-bug count:
  project  total_bugs  reopened
   AMBARI       16600       835
    SPARK       12638       720
CASSANDRA        7885       568
    FLINK        6464       506
    OFBIZ        4083       391

Projects with < 10 reopened bugs: 2
project  reopened
 CTAKES         8
  BAHIR         5


In [11]:
### Saving the summary

In [12]:
out_path = os.path.join(DATA_DIR, "project_summary.csv")
summary.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: C:\Users\ramku\OneDrive\Documents\Research Practicum\Execution\Reopened bugs dataset\project_summary.csv
